In [ ]:
"""
SimCLR 对比学习预训练入口（ main.ipynb ）
- 核心逻辑已剥离至 lib/ 目录下以提高可维护性
- 本文件只保留运行入口和 tqdm 进度包装
"""

import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

from lib.models import SpectrumEncoder, ProjectionHead, SimCLR
from lib.augmentation import SpectrumAugmentation
from lib.utils import ConvergenceChecker, plot_loss_history
from lib.trainer import train_step

# ==================== 配置区 ====================
base_dir = Path(__file__).parent if '__file__' in locals() else Path.cwd()
data_path = base_dir / "pretrain_spectra.npy"
output_dir = base_dir / "pretrained_model_v2"
output_dir.mkdir(parents=True, exist_ok=True)

batch_size = 256
temperature = 0.07
lr = 0.001
weight_decay = 1e-5
patience = 30
max_epochs = 500
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==================== 数据准备 ====================
if not data_path.exists():
    raise FileNotFoundError(f"错误: 未找到数据文件 {data_path}")

print("=" * 60)
print("SimCLR 对比学习预训练（优化版 - 重构入口）")
print("=" * 60)
print(f"\n加载数据: {data_path}")
spectra = np.load(data_path)
print(f"  谱图数量: {spectra.shape[0]:,}")
print(f"  向量维度: {spectra.shape[1]}")

spectra_tensor = torch.tensor(spectra, dtype=torch.float32)
dataset = TensorDataset(spectra_tensor)

actual_batch_size = min(batch_size, max(32, len(dataset) // 10))
print(f"  批次大小: {actual_batch_size}")

dataloader = DataLoader(
    dataset,
    batch_size=actual_batch_size,
    shuffle=True,
    num_workers=4 if device == 'cuda' else 0,
    pin_memory=(device == 'cuda'),
    drop_last=True,
    prefetch_factor=2 if device == 'cuda' else None
)

# ==================== 初始化 ====================
device = torch.device(device)
print(f"\n设备: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    scaler = torch.amp.GradScaler('cuda')
else:
    scaler = None

encoder = SpectrumEncoder(input_dim=spectra.shape[1], hidden_dim=256).to(device)
projection_head = ProjectionHead(input_dim=256, hidden_dim=128, output_dim=64).to(device)
model = SimCLR(encoder, projection_head).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=2, eta_min=lr * 0.001
)

augmenter = SpectrumAugmentation(
    intensity_jitter_range=(0.6, 1.5),
    mz_shift_range=1,
    minor_peak_threshold_ratio=0.20,
    minor_peak_removal_prob=0.30,
    min_peaks_retain=3,
    rayleigh_scale_tolerance=(0.005, 0.05),
    n_clusters_range=(1, 5),
    cluster_width_range=(3, 11),
    cluster_amp_ratio_range=(0.10, 0.50)
)

convergence_checker = ConvergenceChecker(
    patience=patience,
    min_delta=1e-4,
    window_size=10
)

# ==================== 训练循环 (tqdm 包装) ====================
print(f"\n{'='*60}")
print(f"开始训练（自动检测拟合，最多{max_epochs}轮）")
print(f"{'='*60}\n")

loss_history = []
best_loss = float('inf')
start_time = datetime.now()

for epoch in range(1, max_epochs + 1):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch:3d}/{max_epochs}')
    for (batch_data,) in pbar:
        spec = batch_data.to(device, non_blocking=True)
        loss_val = train_step(model, spec, optimizer, augmenter, device, scaler, temperature)
        
        epoch_loss += loss_val
        n_batches += 1
        
        pbar.set_postfix({
            'loss': f'{loss_val:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })
    
    avg_loss = epoch_loss / n_batches
    loss_history.append(avg_loss)
    scheduler.step()
    
    elapsed = datetime.now() - start_time
    print(f'Epoch {epoch:3d} | Loss: {avg_loss:.6f} | '
          f'LR: {scheduler.get_last_lr()[0]:.2e} | '
          f'耗时: {str(elapsed).split(".")[0]}')
    
    # 保存最佳模型
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'encoder_state_dict': encoder.state_dict(),
            'loss': avg_loss,
        }, str(output_dir / 'best_model.pt'))
        print(f'  ✓ 最佳模型 (loss: {avg_loss:.6f})')
    
    # 检测收敛
    if convergence_checker.update(avg_loss):
        print(f'\n{"="*60}')
        print(f'模型在第 {epoch} 轮收敛，停止训练')
        print(f"{'='*60}")
        break

# === 保存最终模型 ===
torch.save(encoder.state_dict(), str(output_dir / 'pretrained_encoder_final.pt'))
print(f"\n编码器已保存: {output_dir / 'pretrained_encoder_final.pt'}")

# === 绘制损失曲线 ===
plot_loss_history(loss_history, best_loss, output_dir / 'pretraining_loss.png')

# === 训练总结 ===
total_time = datetime.now() - start_time
print(f"\n{'='*60}")
print("训练完成")
print(f"{'='*60}")
print(f"  总轮数: {len(loss_history)}")
print(f"  总耗时: {str(total_time).split('.')[0]}")
print(f"  最佳损失: {best_loss:.6f}")
print(f"  最终损失: {loss_history[-1]:.6f}")
print(f"  模型位置: {output_dir}")
print(f"{'='*60}")

SimCLR 对比学习预训练（优化版 - 重构入口）

加载数据: D:\UserFiles\Documents\PyCharm\SimCLR\pretrain_spectra.npy
  谱图数量: 331,558
  向量维度: 561
  批次大小: 256

设备: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
模型参数量: 248,768

开始训练（自动检测拟合，最多500轮）



Epoch   1/500:   0%|          | 6/1295 [00:09<27:44,  1.29s/it, loss=6.2160, lr=1.00e-03]  